In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount.
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/LCX_Source_Anatomy_Topology_Validation_v1'
BRANCH='lcx-source-anatomy-topology-validation-from-main'


# OpenPlaque — LCX source-space anatomy/topology validation

This experiment ranks the five strongest previously nominated source paths using **source-space anatomy/topology only**. Prior curved-template LCX scores are supporting metadata and are excluded from the anatomy score and gates. LCX remains unresolved unless later independently confirmed.


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/total/aorta.nii.gz',
 root/'Joint_Three_Vessel_Template_Classifier_v1/LCX_joint_candidate_ranking.csv',
]+[root/f'Joint_Three_Vessel_Template_Classifier_v1/candidate_{i:02d}_source_path.csv' for i in range(1,6)]
missing=[str(p) for p in required if not p.exists()]
print(f'INPUT PREFLIGHT: {len(required)-len(missing)}/{len(required)} present')
for p in required: print(('OK   ' if p.exists() else 'MISS '),p)
if missing: raise FileNotFoundError('Missing required cached inputs:\n'+'\n'.join(missing))


In [ ]:
import shutil, subprocess, sys, textwrap
repo='/content/OpenPlaque'
shutil.rmtree(repo, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest'], check=True)
print('COMMIT:')
subprocess.run(['git','-C',repo,'rev-parse','HEAD'], check=True)
print('IMPORT CHECK:')
subprocess.run([sys.executable,'-c','import openplaque; print(openplaque.__file__)'], check=True)
print('TESTS:')
subprocess.run([sys.executable,'-m','pytest','-q',repo+'/tests/test_lcx_source_anatomy_topology_validation.py'], check=True)


In [ ]:
runner=textwrap.dedent(f'''
from openplaque.lcx_source_anatomy_topology_validation import synthetic_anatomy_topology_self_test, run
print('SELF TEST:', synthetic_anatomy_topology_self_test(), flush=True)
result=run({DRIVE_ROOT!r},{OUTPUT_ROOT!r})
print('STATUS:',result['summary']['status'], flush=True)
print('REPORT:',result['report'], flush=True)
print('ZIP:',result['zip'], flush=True)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_SOURCE_ANATOMY_TOPOLOGY_VALIDATION_REPORT_BACK.zip', flush=True)
''')
proc=subprocess.run([sys.executable,'-c',runner],text=True,capture_output=True)
print(proc.stdout)
if proc.stderr:
    print('--- WORKFLOW STDERR ---')
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'LCX anatomy/topology workflow failed with exit code {proc.returncode}; full traceback is printed above')
